# Option pricing and delta hedging

European call, no dividends. Reference contract used below: spot 100, strike 100, rate 0.05. Volatility is a decimal.

The four figures are also saved under `figures/` and shown in the README.


## Greeks against the spot

Call delta, gamma, and vega for four maturities. The vertical line is the strike. Gamma and vega of the put are the same; the put delta is the call delta minus one.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from greeks import delta_call, gamma_call, vega

K, r, sigma = 100.0, 0.05, 0.2
spots = np.linspace(50.0, 150.0, 400)
maturities = (
    (7 / 365, "1 week"),
    (30 / 365, "1 month"),
    (0.25, "3 months"),
    (1.0, "1 year"),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True)
for T, label in maturities:
    axes[0].plot(spots, delta_call(spots, K, T, r, sigma), label=label)
    axes[1].plot(spots, gamma_call(spots, K, T, r, sigma), label=label)
    axes[2].plot(spots, vega(spots, K, T, r, sigma), label=label)

axes[0].set_title("Call delta")
axes[1].set_title("Gamma")
axes[2].set_title("Vega")
for ax in axes:
    ax.set_xlabel("Spot")
    ax.axvline(K, color="grey", linewidth=0.8, linestyle="--")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Greek")
fig.tight_layout()
plt.show()


## Hedge error against the number of rebalances

Short call, realized volatility equal to implied volatility (both 0.20). Each point is the standard deviation of the maturity P&L over 5,000 paths. The dashed line goes through the first point with slope -1/2, which is the \(1/\sqrt{N}\) decay of the discrete-hedge residual.


In [ ]:
from hedge_sim import hedge_pnl

S0, K, T, r = 100.0, 100.0, 1.0, 0.05
sigma_i = sigma_r = 0.2
n_steps = np.array([4, 12, 26, 52, 252])
n_paths, seed = 5_000, 0

stds = np.array([
    np.std(hedge_pnl(S0, K, T, r, sigma_i, sigma_r, int(n), n_paths, seed))
    for n in n_steps
])
std_ref = stds[0] * np.sqrt(n_steps[0] / n_steps)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(n_steps, stds, "o", label="simulated standard deviation")
ax.loglog(n_steps, std_ref, "--", label="slope -1/2")
ax.set_xlabel("Number of steps")
ax.set_ylabel("P&L standard deviation")
ax.legend()
fig.tight_layout()
plt.show()

for n, s in zip(n_steps, stds):
    print(f"N={n:3d}  std={s:.3f}")


## Mean P&L against the volatility gap

Same short call, 52 steps, implied volatility fixed at 0.20. The horizontal axis is realized volatility minus implied volatility. The trader chooses the implied volatility. The realized volatility is what the path does afterwards.


In [ ]:
sigma_i = 0.2
n_steps_fixed, n_paths, seed = 52, 5_000, 0
sigma_r = np.linspace(0.10, 0.30, 9)

means = np.array([
    np.mean(hedge_pnl(S0, K, T, r, sigma_i, float(sr), n_steps_fixed, n_paths, seed))
    for sr in sigma_r
])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sigma_r - sigma_i, means, "o-")
ax.axhline(0.0, color="grey", linewidth=0.8, linestyle="--")
ax.axvline(0.0, color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("sigma_r - sigma_i")
ax.set_ylabel("Mean P&L")
ax.set_title("Short call, 52 steps, sigma_i = 0.20")
fig.tight_layout()
plt.show()

for gap, m in zip(sigma_r - sigma_i, means):
    print(f"sigma_r - sigma_i = {gap:+.3f}  mean = {m:+.3f}")


## Transaction costs against hedge frequency

Each share trade, including the opening hedge and the final unwind, costs `k * |shares traded| * spot`. `k = 0.001` is 10 basis points of traded notional. Realized and implied volatility are both 0.20, so the only thing moving the mean is the cost.


In [ ]:
n_paths, seed = 5_000, 0
costs = (0.0, 0.0005, 0.001, 0.002)

fig, ax = plt.subplots(figsize=(6, 4))
for k in costs:
    row = np.array([
        np.mean(hedge_pnl(S0, K, T, r, sigma_i, sigma_i, int(n), n_paths, seed, k))
        for n in n_steps
    ])
    ax.plot(n_steps, row, "o-", label=f"k = {k:g}")

ax.axhline(0.0, color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("Number of steps")
ax.set_ylabel("Mean net P&L")
ax.set_title("Short call, sigma_r = sigma_i = 0.20")
ax.legend()
fig.tight_layout()
plt.show()
